[구글 코랩(Colab)에서 실행하기](https://colab.research.google.com/github/lovedlim/bae-v3/blob/main/part4/ch11/p11_type3.ipynb)

## 작업형3

### 문제1

1-1. 범주형 독립변수의 오즈비(Odds Ratio) 구하기

학습 데이터(Train)를 이용하여 로지스틱 회귀분석을 수행하시오. 구축된 모형을 바탕으로 다른 독립변수들이 고정되어 있을 때, **당뇨 여부(diabetic)가 'yes'인 집단의 오즈비(Odds Ratio)**를 구하시오.

정답은 소수점 셋째 자리에서 반올림하여 소수점 둘째 자리까지 출력하시오.

In [9]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch11/elderly_health.csv")
df

train = df[:2000].copy()
test = df[2000:].copy()

In [10]:
train

,id,age,diabetic,activity,glus_fast,bmi,blood_pressure,predicted
0,1,70,no,no,154,26.8,136,1
1,2,58,no,yes,113,23.7,136,0
2,3,55,no,no,156,25.5,141,0
3,4,50,no,yes,183,29.8,100,0
4,5,40,no,no,122,29.9,125,1
...,...,...,...,...,...,...,...,...
1995,1996,63,no,yes,105,24.3,129,0
1996,1997,41,pre,yes,94,27.7,114,0
1997,1998,56,no,yes,160,28.2,111,0
1998,1999,60,no,yes,95,31.5,131,0


In [11]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              2000 non-null   int64  
 1   age             2000 non-null   int64  
 2   diabetic        2000 non-null   object 
 3   activity        2000 non-null   object 
 4   glus_fast       2000 non-null   int64  
 5   bmi             2000 non-null   float64
 6   blood_pressure  2000 non-null   int64  
 7   predicted       2000 non-null   int64  
dtypes: float64(1), int64(5), object(2)
memory usage: 125.1+ KB


In [12]:
train['diabetic'].value_counts()

,count
diabetic,
no,1200
pre,526
yes,274


In [23]:
import statsmodels.formula.api as smf
import numpy as np

#train = pd.get_dummies(train)
formula = "predicted ~ age + diabetic + activity + glus_fast + bmi + blood_pressure"
model = smf.logit(formula, data = train).fit()
#print(model.summary())
#print(model.params)
#print("diabetic[T.yes] : ", model.params[2])
coef_diabetic_yes = model.params["diabetic[T.yes]"]
result = np.exp(coef_diabetic_yes)
print(round(result,2))

Optimization terminated successfully.
         Current function value: 0.486192
         Iterations 7
20.36


In [ ]:
# 풀이1-1
import pandas as pd
import numpy as np
from statsmodels.formula.api import logit

df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch11/elderly_health.csv")

# 1) Train/Test 데이터 분할
train = df[:2000].copy()
test = df[2000:].copy()

# 2) 로지스틱 회귀분석 수행
model = logit("predicted ~ age + diabetic + activity + glus_fast + bmi + blood_pressure",
              data=train).fit()

# 3) 회귀 분석 결과 출력
print(model.summary())

# 4) diabetic='yes'의 오즈비 계산 및 출력
odds_ratio = np.exp(model.params["diabetic[T.yes]"])
print(round(odds_ratio, 2))

# 정답: 20.36

Optimization terminated successfully.
         Current function value: 0.486192
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              predicted   No. Observations:                 2000
Model:                          Logit   Df Residuals:                     1992
Method:                           MLE   Df Model:                            7
Date:                Sat, 03 Jan 2026   Pseudo R-squ.:                  0.2883
Time:                        06:11:30   Log-Likelihood:                -972.38
converged:                       True   LL-Null:                       -1366.3
Covariance Type:            nonrobust   LLR p-value:                7.486e-166
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -6.7577      0.754     -8.960      0.000      -8.236      -5.279
diabetic[T.pre

1-2. 최대 확률값에 해당하는 데이터 조건 탐색

문제 1-1에서 구축한 회귀모형을 검증 데이터(Test)에 적용하여 각 데이터별 발병 확률을 예측하시오. 예측된 확률값 중 가장 높은 발병 확률을 가진 대상(행)의 공복 혈당(glus_fast) 값을 구하시오.

정답은 정수 형태로 출력하시오.

In [44]:
max_idx = model.predict(test).idxmax()
test.loc[max_idx]['glus_fast']

np.int64(181)

In [ ]:
# 풀이1-2
# 1) test 데이터 예측 확률 계산
pred = model.predict(test)

# 2) 최대 확률을 가진 데이터의 인덱스 찾기
max_idx = pred.idxmax()

# # 3) 해당 데이터의 glus_fast 값 출력
print(test.loc[max_idx])

# 정답: 181

id                2153
age                 78
diabetic           pre
activity            no
glus_fast          181
bmi               26.3
blood_pressure     115
predicted            1
Name: 2152, dtype: object


1-3. 임계값 조건에 따른 민감도(Sensitivity) 계산

검증 데이터(Test)에 대한 예측 확률값에 대해 임계값(Threshold)을 0.2로 설정하여, 확률이 0.2를 초과하면 1(발병), 그렇지 않으면 0(정상)으로 분류하시오. 이 분류 결과를 바탕으로 검증 데이터의 **민감도(Sensitivity, 재현율)**를 계산하시오.

정답은 소수점 셋째 자리에서 반올림하여 소수점 둘째 자리까지 출력하시오.

In [58]:
pred = model.predict(test)
result =  pred.apply(lambda x: int(x> 0.2))
#print(result)

#print("재현율 = TP/TP + FN")

from sklearn.metrics import recall_score
#print(help(recall_score))
sensitivity = recall_score(test['predicted'], result)
print(round(sensitivity,2))


0.94


In [ ]:
# 풀이1-3
from sklearn.metrics import recall_score

# 1) test 데이터 예측
pred = model.predict(test)

# 2) 임계값 0.2 적용하여 분류
pred = (pred > 0.2).astype(int)

# 3) 민감도 계산 및 출력
sensitivity = recall_score(test['predicted'], pred)
print(round(sensitivity, 2))

# 정답: 0.94

0.94


### 문제2

2-1. 단일표본 t-검정 (One-sample t-test)

기존 마케팅 방식의 평균 매출액은 35,000으로 알려져 있다. 이번 프로모션을 통해 얻은 매출액(sales)의 평균이 기존 평균인 35,000과 다르다고 할 수 있는지 단일표본 t-검정을 수행하고, 이때의 **p-value(유의확률)**를 구하시오.

정답은 소수점 넷째 자리에서 반올림하여 소수점 셋째 자리까지 출력하시오.

In [60]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch11/promotion_data.csv")
df

,sales,visit_count,page_time,ad_clicks,promotion_budget
0,35341.611727,36,168,18,11340
1,64757.849132,32,214,21,11224
2,40615.684241,56,172,6,5468
3,52009.480757,70,234,5,7486
4,32462.259065,27,88,8,7547
...,...,...,...,...,...
195,46537.182799,27,131,8,4121
196,31829.997843,21,142,5,5155
197,24390.270081,35,86,29,11870
198,43025.977501,30,244,28,11433


In [67]:
from scipy import stats
st, pv = stats.ttest_1samp(df['sales'], 35000)
print(st)
print(round(pv, 3))
#print(help(stats.ttest_1samp))

1.3826844415882558
0.168


In [ ]:
import pandas as pd
from scipy.stats import ttest_1samp

df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch11/promotion_data.csv")

# 1) 단일표본 t검정 수행
t_stat, p_value = ttest_1samp(df['sales'], 35000)

# 2) p-value 추출 및 출력
print(round(p_value, 3))

# 정답: 0.168

0.168


2-2. 변수 간 상관관계 분석

데이터셋 내의 수치형 변수들 간의 피어슨 상관계수를 구하고자 한다. 전체 변수 간의 상관계수 행렬을 구한 뒤 결과를 확인하시오. (※ 실제 시험에서는 특정 두 변수 간의 상관계수 수치를 단일값으로 묻는 형태로 출제됩니다. 예: sales와 promotion_budget 간의 상관계수)

정답은 소수점 넷째 자리에서 반올림하여 소수점 셋째 자리까지 출력하시오.

In [77]:
#print(help(df.corr))
#round(df.corr(method = 'pearson').loc['sales', 'promotion_budget'],3)
df.corr()

,sales,visit_count,page_time,ad_clicks,promotion_budget
sales,1.000000,0.044154,0.575396,-0.131915,-0.091270
visit_count,0.044154,1.000000,0.011349,0.078189,-0.089217
page_time,0.575396,0.011349,1.000000,-0.093688,-0.039554
ad_clicks,-0.131915,0.078189,-0.093688,1.000000,0.036940
promotion_budget,-0.091270,-0.089217,-0.039554,0.036940,1.000000


In [ ]:
# 1) 상관계수 행렬 계산
print(df.corr())

# 정답: 0.505

                     sales  visit_count  page_time  ad_clicks  \
sales             1.000000     0.044154   0.575396  -0.131915   
visit_count       0.044154     1.000000   0.011349   0.078189   
page_time         0.575396     0.011349   1.000000  -0.093688   
ad_clicks        -0.131915     0.078189  -0.093688   1.000000   
promotion_budget -0.091270    -0.089217  -0.039554   0.036940   

                  promotion_budget  
sales                    -0.091270  
visit_count              -0.089217  
page_time                -0.039554  
ad_clicks                 0.036940  
promotion_budget          1.000000  


2-3. 다중 회귀분석과 통계적 유의성 검정

종속변수를 매출액(sales)으로 하고, 독립변수를 방문자 수(visit_count), 페이지 체류 시간(page_time), 광고 클릭 수(ad_clicks), 프로모션 예산(promotion_budget)으로 하는 다중 선형 회귀분석을 수행하시오.

구축된 회귀모형에서 **통계적으로 가장 유의한 변수(p-value가 가장 작은 변수)의 회귀계수(Coefficient)**를 구하시오.

정답은 소수점 넷째 자리에서 반올림하여 소수점 셋째 자리까지 출력하시오.

In [83]:
import statsmodels.formula.api as smf
df
formula = "sales ~ visit_count +  page_time + ad_clicks + promotion_budget"
model = smf.ols(formula, data = df).fit()
#print(model.summary())
#print(model.pvalues.idxmin())
idx_min = model.pvalues.idxmin()
print(round(model.params[idx_min],3))
print()



130.739



In [ ]:
from statsmodels.formula.api import ols

# 1) ols 회귀분석 수행
model = ols("sales ~ visit_count + page_time + ad_clicks + promotion_budget", data=df).fit()

# 2) 회귀분석 결과 확인
print(model.summary())

# 3) p-value가 가장 작은 변수 찾기
min_var = model.pvalues.idxmin()
print("가장 유의한 변수:", min_var)

# 4) 해당 변수의 회귀계수 출력
coef = model.params[min_var]
print(round(coef, 3))

# 정답: 130.739

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.343
Model:                            OLS   Adj. R-squared:                  0.330
Method:                 Least Squares   F-statistic:                     25.45
Date:                Sat, 03 Jan 2026   Prob (F-statistic):           5.60e-17
Time:                        06:46:50   Log-Likelihood:                -2160.2
No. Observations:                 200   AIC:                             4330.
Df Residuals:                     195   BIC:                             4347.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept         2.234e+04   4136.622  